# Notebook 01b — New Spotify Dataset: EDA, Cleaning, Feature Engineering, PCA

Replaces `tracks_features.csv` (1.2M rows, audio features, **no popularity**) with a
new source: [maharshipandya/spotify-tracks-dataset](https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset)
(114,000 rows) — smaller, but has a real `popularity` column and a `track_genre` label
the old dataset never had.

This notebook mirrors `01_spotify_eda.ipynb` (EDA/cleaning) and the Spotify half of
`04_pca_.ipynb` (feature engineering + PCA) back to back, applied to the new source.
`popularity` is carried through as metadata only — per plan, it's used to filter/rank
recommendations at recommend-time, not fed into the PCA itself.

Outputs (kept separate from the originals so nothing currently working is overwritten):
- `data/processed/tracks_features_new_clean.csv`
- `data/processed/spotify_scaler_new.pkl`
- `data/processed/spotify_pca_new.pkl`
- `data/processed/song_pca_new.csv`


In [1]:
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

RAW = r"C:\Users\ryanm\Documents\coding_projects\restaurant_recommendation\data\raw"
PROCESSED = r"C:\Users\ryanm\Documents\coding_projects\restaurant_recommendation\data\processed"

df = pd.read_csv(os.path.join(RAW, "tracks_features_new.csv"))
print(f"Raw shape: {df.shape}")
df.head(3)


Raw shape: (114000, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,...,-6.746,0,0.1430,0.0322,0.000001,0.358,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.166,...,-17.235,1,0.0763,0.9240,0.000006,0.101,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.359,...,-9.734,1,0.0557,0.2100,0.000000,0.117,0.120,76.332,4,acoustic


## Step 1 — EDA: what's actually wrong with this file?

Same questions as notebook 01, asked of the new source.


In [2]:
print("Nulls per column (only columns with any):")
print(df.isna().sum()[df.isna().sum() > 0])
print()
print("Unique track_id:", df["track_id"].nunique(), "/", len(df))
print("Rows involved in (track_name, artists) duplicates:",
      df.duplicated(subset=["track_name", "artists"], keep=False).sum())
print("tempo == 0:", (df["tempo"] == 0).sum())
print("duration_ms == 0:", (df["duration_ms"] == 0).sum())
print()
print("popularity distribution:")
print(df["popularity"].describe())
print("popularity == 0:", (df["popularity"] == 0).sum(), f"({(df['popularity'] == 0).mean():.1%})")


Nulls per column (only columns with any):
artists       1
album_name    1
track_name    1
dtype: int64

Unique track_id: 89741 / 114000


Rows involved in (track_name, artists) duplicates: 49157
tempo == 0: 157
duration_ms == 0: 1

popularity distribution:
count    114000.000000
mean         33.238535
std          22.305078
min           0.000000
25%          17.000000
50%          35.000000
75%          50.000000
max         100.000000
Name: popularity, dtype: float64
popularity == 0: 16020 (14.1%)


## Step 2 — Clean

Same fixes as the original dataset's known issues, applied here:
- drop null rows (1 row — also has `duration_ms == 0`, a broken entry either way)
- drop `tempo == 0` rows (not a real tempo — same rule as `CLAUDE.md`'s documented issue)
- dedupe on `(track_name, artists)`, keep first occurrence (same key used everywhere
  else in this project, e.g. `songs.drop_duplicates(subset=['name','artists'])` in 05b)
- cast `explicit` from bool to int (ML-ready, same convention as the Yelp binaries)


In [3]:
before = len(df)

df = df.dropna(subset=["artists", "album_name", "track_name"])
df = df[df["tempo"] != 0]
df = df[df["duration_ms"] != 0]
df["explicit"] = df["explicit"].astype(int)

n_before_dedup = len(df)
df = df.drop_duplicates(subset=["track_name", "artists"], keep="first").reset_index(drop=True)

print(f"{before:,} raw rows")
print(f"-> {n_before_dedup:,} after dropping nulls / tempo==0 / duration==0")
print(f"-> {len(df):,} after deduping on (track_name, artists)")

df.to_csv(os.path.join(PROCESSED, "tracks_features_new_clean.csv"), index=False)


114,000 raw rows
-> 113,842 after dropping nulls / tempo==0 / duration==0
-> 81,198 after deduping on (track_name, artists)


## Step 3 — Feature engineering: select, check skew, log1p, standardize

Identical treatment to notebook 04's Spotify half — same 9 continuous audio features,
same 4 columns get `log1p` (their skew was > 1 in the original dataset too), same
`StandardScaler`. `popularity` and `track_genre` are deliberately excluded from this
feature set — they ride along as metadata on `song_pca_new.csv`, not as PCA inputs.


In [4]:
AUDIO_FEATURES = [
    "danceability", "energy", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "loudness", "tempo",
]

df_audio = df[AUDIO_FEATURES].copy()

print("Skew per feature:")
print(df_audio.skew().sort_values(ascending=False).round(2))


Skew per feature:
speechiness         4.48
liveness            2.04
instrumentalness    1.47
acousticness        0.65
tempo               0.28
valence             0.15
danceability       -0.36
energy             -0.56
loudness           -1.92
dtype: float64


In [5]:
SKEWED = ["instrumentalness", "acousticness", "speechiness", "liveness"]

df_audio[SKEWED] = df_audio[SKEWED].apply(np.log1p)

print("Skew after log1p:")
print(df_audio[SKEWED].skew().round(2))


Skew after log1p:
instrumentalness    1.39
acousticness        0.48
speechiness         3.64
liveness            1.69
dtype: float64


In [6]:
scaler_s = StandardScaler()
X_scaled = scaler_s.fit_transform(df_audio)
print(f"Scaled matrix: {X_scaled.shape}")


Scaled matrix: (81198, 9)


## Step 4 — Fit PCA (same 85%-variance rule as notebook 04)


In [7]:
pca_s = PCA()
pca_s.fit(X_scaled)
cumvar = np.cumsum(pca_s.explained_variance_ratio_)
for i, cv in enumerate(cumvar, 1):
    print(f" {i} components -> {cv:.1%} variance")


 1 components -> 31.6% variance
 2 components -> 47.9% variance
 3 components -> 61.7% variance
 4 components -> 71.8% variance
 5 components -> 81.5% variance
 6 components -> 89.7% variance
 7 components -> 94.8% variance
 8 components -> 98.5% variance
 9 components -> 100.0% variance


In [8]:
pca_s_final = PCA(n_components=0.85)
song_pca_mat = pca_s_final.fit_transform(X_scaled)

print(f"Using {pca_s_final.n_components_} components "
      f"({sum(pca_s_final.explained_variance_ratio_):.1%} variance retained)")
print(f"song_pca_mat shape: {song_pca_mat.shape}")


Using 6 components (89.7% variance retained)
song_pca_mat shape: (81198, 6)


## Step 5 — What does each component represent?


In [9]:
loadings_s = pd.DataFrame(
    pca_s_final.components_,
    index=[f"pc{i+1}" for i in range(pca_s_final.n_components_)],
    columns=AUDIO_FEATURES,
)
print(loadings_s.round(2).to_string())


     danceability  energy  speechiness  acousticness  instrumentalness  liveness  valence  loudness  tempo
pc1          0.24    0.51         0.12         -0.43             -0.28      0.10     0.29      0.52   0.20
pc2          0.56   -0.27         0.07          0.34             -0.34     -0.18     0.53     -0.07  -0.26
pc3         -0.16    0.00         0.61          0.22             -0.22      0.69    -0.05     -0.05  -0.17
pc4          0.35    0.16         0.40         -0.29              0.53     -0.15    -0.13     -0.05  -0.53
pc5          0.18   -0.08         0.49          0.09              0.28     -0.16     0.11     -0.20   0.74
pc6          0.17    0.06        -0.41         -0.00              0.49      0.59     0.43     -0.17   0.06


## Step 6 — Save: `song_pca_new.csv` + scaler/PCA artifacts

`popularity` and `track_genre` are attached here as metadata columns alongside the PC
scores — available for filtering/ranking at recommend-time, never used as PCA inputs.


In [10]:
song_pca_df = pd.DataFrame(
    song_pca_mat,
    columns=[f"pc{i+1}" for i in range(pca_s_final.n_components_)],
)
song_pca_df.insert(0, "id", df["track_id"].values)
song_pca_df.insert(1, "name", df["track_name"].values)
song_pca_df.insert(2, "artists", df["artists"].values)
song_pca_df["popularity"] = df["popularity"].values
song_pca_df["track_genre"] = df["track_genre"].values

song_pca_df.to_csv(os.path.join(PROCESSED, "song_pca_new.csv"), index=False)
joblib.dump(scaler_s, os.path.join(PROCESSED, "spotify_scaler_new.pkl"))
joblib.dump(pca_s_final, os.path.join(PROCESSED, "spotify_pca_new.pkl"))

print(f"song_pca_new.csv         {song_pca_df.shape}")
song_pca_df.head(3)


song_pca_new.csv         (81198, 11)


,id,name,artists,pc1,pc2,pc3,pc4,pc5,pc6,popularity,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Comedy,Gen Hoshino,0.741021,1.108196,0.882007,0.656191,-0.738783,0.315983,73,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ben Woodward,-3.120386,1.018622,0.491047,-0.300469,-0.816842,-1.002791,55,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,Ingrid Michaelson;ZAYN,-1.348065,-0.185247,-0.045223,0.322937,-1.542504,-1.269471,57,acoustic
